# 🏦 Treasury Bulletin RAG — Baseline vs. Engineered

**Author:** Yifan Wang
**Years covered:** 2018–2025
**Data source:** [Databricks OfficeQA](https://huggingface.co/datasets/databricks/officeqa)

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline over U.S. Treasury
Bulletins and compares a **Baseline** configuration against an **Engineered** configuration,
using the metrics required by the Self-Discovery Lab (Hit Rate@5, MRR, Groundedness,
Factual Accuracy, Hallucination Rate).

Unlike a naive script-per-cell notebook, this version is organized around small,
reusable classes (`Chunker`, `VectorIndex`, `Generator`) so the baseline and engineered
runs are just two different **configs** passed through the same pipeline — that makes
the comparison apples-to-apples and the code easy to extend later.

**Architecture at a glance**

| Component | Baseline | Engineered |
|---|---|---|
| Vector index | FAISS (flat, cosine) | FAISS (flat, cosine) |
| Embedding model | `multi-qa-MiniLM-L6-cos-v1` | `multi-qa-MiniLM-L6-cos-v1` |
| Chunk size | 400 chars | 1200 chars |
| Overlap | 0 | 250 chars |
| Metadata | none | `year`, `month`, `quarter` |
| Retrieval filter | pure semantic | metadata pre-filter + semantic |
| Generator | Gemini (model auto-fallback) | Gemini (model auto-fallback) |
| Prompt | strict, terse | guided, table-aware |


## 1. Setup

In [ ]:
!pip install -q huggingface_hub pandas faiss-cpu sentence-transformers google-genai tqdm

In [ ]:
from huggingface_hub import snapshot_download, login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

snapshot_download(
    repo_id="databricks/officeqa",
    repo_type="dataset",
    allow_patterns="treasury_bulletins_parsed/transformed/*.txt",
    local_dir="./data"
)
snapshot_download(
    repo_id="databricks/officeqa",
    repo_type="dataset",
    allow_patterns="officeqa_full.csv",
    local_dir="./data"
)
print("Data ready.")

## 2. Config

Everything that differs between Baseline and Engineered lives in one dict each,
so the rest of the pipeline never branches on "which run is this."


In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class RunConfig:
    name: str
    chunk_size: int
    overlap: int
    use_metadata: bool
    embedding_model: str = "multi-qa-MiniLM-L6-cos-v1"
    top_k: int = 5

BASELINE = RunConfig(name="baseline", chunk_size=400, overlap=0, use_metadata=False)
ENGINEERED = RunConfig(name="engineered", chunk_size=1200, overlap=250, use_metadata=True)

YEARS = [str(y) for y in range(2018, 2026)]
TXT_DIR = "./data/treasury_bulletins_parsed/transformed"


## 3. Build the question set (answer key filtered to our years)

In [ ]:
import pandas as pd

def load_question_set(years):
    df = pd.read_csv("./data/officeqa_full.csv")
    pattern = '|'.join(years)
    touching = df[df['source_files'].str.contains(pattern, na=False)].copy()

    def all_in_years(s):
        if pd.isna(s):
            return False
        files = [f.strip() for f in str(s).split('\n') if f.strip()]
        return all(any(y in f for y in years) for f in files)

    final = touching[touching['source_files'].apply(all_in_years)].copy()
    return final.reset_index(drop=True)

df_questions = load_question_set(YEARS)
df_questions.to_csv("./data/filtered_questions.csv", index=False)
print(f"Questions fully answerable from {YEARS[0]}-{YEARS[-1]}: {len(df_questions)}")
print(df_questions['difficulty'].value_counts())


## 4. Chunking

Two strategies, one interface: `Chunker.build(files)` -> list of chunk dicts.

In [ ]:
import os, re, glob

def extract_year_month(filename):
    m = re.search(r'(\d{4})_(\d{2})', filename)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

def month_to_quarter(month):
    if month is None:
        return None
    return (month - 1) // 3 + 1

class Chunker:
    def __init__(self, cfg: RunConfig):
        self.cfg = cfg

    def build(self, files):
        chunks = []
        step = self.cfg.chunk_size - self.cfg.overlap
        for filepath in files:
            filename = os.path.basename(filepath)
            year, month = extract_year_month(filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                text = f.read()
            for i in range(0, len(text), step):
                piece = text[i:i + self.cfg.chunk_size]
                if len(piece.strip()) < 50:
                    continue
                meta = {"source": filename}
                if self.cfg.use_metadata:
                    meta.update({"year": year, "month": month, "quarter": month_to_quarter(month)})
                chunks.append({
                    "text": piece,
                    "meta": meta,
                    "chunk_id": f"{filename}_{i}"
                })
        return chunks

def list_bulletin_files(years):
    files = []
    for y in years:
        files.extend(glob.glob(os.path.join(TXT_DIR, f"treasury_bulletin_{y}_*.txt")))
    return sorted(files)

bulletin_files = list_bulletin_files(YEARS)
print(f"Found {len(bulletin_files)} bulletin files across {YEARS[0]}-{YEARS[-1]}")


## 5. Vector index (FAISS)

A thin wrapper so retrieval code doesn't care about the underlying library.

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

class VectorIndex:
    def __init__(self, embedder):
        self.embedder = embedder
        self.index = None
        self.chunks = []

    def build(self, chunks, batch_size=256):
        self.chunks = chunks
        texts = [c["text"] for c in chunks]
        vecs = self.embedder.encode(texts, batch_size=batch_size, show_progress_bar=True,
                                     normalize_embeddings=True)
        vecs = np.asarray(vecs, dtype="float32")
        self.index = faiss.IndexFlatIP(vecs.shape[1])  # cosine sim via normalized inner product
        self.index.add(vecs)
        return self

    def search(self, query, k=5, year_filter=None, month_filter=None):
        q_vec = self.embedder.encode([query], normalize_embeddings=True).astype("float32")

        # Metadata pre-filter: only search within the matching subset (falls back to
        # full index if the filter would exclude everything).
        if year_filter is not None:
            candidate_idx = [
                i for i, c in enumerate(self.chunks)
                if c["meta"].get("year") == year_filter
                and (month_filter is None or c["meta"].get("month") == month_filter)
            ]
        else:
            candidate_idx = None

        if candidate_idx:
            sub_index = faiss.IndexFlatIP(self.index.d)
            sub_vecs = np.stack([self.index.reconstruct(i) for i in candidate_idx])
            sub_index.add(sub_vecs)
            scores, local_ids = sub_index.search(q_vec, min(k, len(candidate_idx)))
            hit_ids = [candidate_idx[i] for i in local_ids[0]]
        else:
            scores, ids = self.index.search(q_vec, k)
            hit_ids = [i for i in ids[0] if i != -1]

        return [self.chunks[i] for i in hit_ids]

embedder = SentenceTransformer("sentence-transformers/multi-qa-MiniLM-L6-cos-v1")
print("Embedder ready:", embedder)


## 6. Query-time metadata extraction (year/month, used only by the Engineered run)

In [ ]:
MONTHS = {'january':1,'february':2,'march':3,'april':4,'may':5,'june':6,
          'july':7,'august':8,'september':9,'october':10,'november':11,'december':12}

def parse_query_time(query):
    year_match = re.search(r'\b(20\d{2})\b', query)
    year = int(year_match.group(1)) if year_match else None
    q_lower = query.lower()
    month = next((num for name, num in MONTHS.items() if name in q_lower), None)
    return year, month


## 7. Retriever metrics — Hit Rate@K and MRR

In [ ]:
def get_ground_truth_files(source_str):
    if pd.isna(source_str):
        return set()
    return set(f.strip() for f in str(source_str).split('\n') if f.strip().endswith('.txt'))

def evaluate_retrieval(index: VectorIndex, cfg: RunConfig, questions_df):
    hits, mrr_sum, log = 0, 0.0, []
    for _, row in questions_df.iterrows():
        gt_files = get_ground_truth_files(row['source_files'])
        year, month = (parse_query_time(row['question']) if cfg.use_metadata else (None, None))
        results = index.search(row['question'], k=cfg.top_k, year_filter=year, month_filter=month)

        seen, ordered_sources = set(), []
        for r in results:
            src = r["meta"]["source"]
            if src not in seen:
                seen.add(src)
                ordered_sources.append(src)

        hit = any(gt in ordered_sources for gt in gt_files)
        hits += hit
        rank = next((i for i, s in enumerate(ordered_sources, 1) if s in gt_files), 0)
        mrr_sum += (1.0 / rank) if rank else 0.0
        log.append({"uid": row['uid'], "hit": hit, "rank": rank})

    n = len(questions_df)
    return {"hit_rate": hits / n, "mrr": mrr_sum / n, "log": log}


## 8. Generator (Gemini, with model fallback + retry)

In [ ]:
from google import genai
from google.colab import userdata
import time, re as _re

gemini_client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

CANDIDATE_MODELS = [
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash-lite",
    "gemini-2.0-flash-001",
    "gemini-flash-lite-latest",
]

PROMPT_TEMPLATE = '''You are a financial data assistant answering questions about U.S. Treasury Bulletins.

Use the context below to find the answer. The answer IS in the context -- look carefully
at tables, numbers, and dates.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
1. Find the specific number, date, or value that answers the question.
2. Look at tables carefully -- the answer may be at a row/column intersection.
3. Match the exact format the question requests (e.g. "$1,234", "March 2020", "1.234").
4. Give your best answer from the context; only say INSUFFICIENT_CONTEXT if the topic
   is completely absent.
5. Output ONLY the final answer, wrapped like this: <answer>your answer here</answer>
'''

def call_gemini(prompt, max_retries=2):
    for model_name in CANDIDATE_MODELS:
        for attempt in range(max_retries):
            try:
                resp = gemini_client.models.generate_content(model=model_name, contents=prompt)
                return resp.text, model_name
            except Exception as e:
                err = str(e)
                if "503" in err or "UNAVAILABLE" in err:
                    time.sleep(3)
                else:
                    break
    return None, None

def extract_answer(llm_text):
    if not llm_text:
        return ""
    m = _re.search(r'<answer>(.*?)</answer>', llm_text, _re.DOTALL)
    if m:
        return m.group(1).strip()
    lines = [l.strip() for l in llm_text.split('\n') if l.strip()]
    return lines[-1] if lines else ""


## 9. Groundedness / hallucination check

In [ ]:
def normalize_number(n):
    s = str(n).replace(',', '').replace('$', '').replace('%', '').strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return s

def extract_numbers(text):
    if not text:
        return []
    clean = str(text).replace(',', '').replace('$', '').replace('%', '')
    return [normalize_number(n) for n in _re.findall(r'-?\d+\.?\d*', clean) if n]

def number_in_context(num, context_clean):
    variants = {num, f"{num}.0", f"{num}.00"}
    return any(v in context_clean for v in variants)

def check_groundedness(prediction, context):
    if not prediction or "INSUFFICIENT_CONTEXT" in prediction:
        return None
    nums = extract_numbers(prediction)
    if not nums:
        return None
    context_clean = context.replace(',', '').replace('$', '').replace('%', '')
    supported = sum(number_in_context(n, context_clean) for n in nums)
    return supported, len(nums)


## 10. Official scorer (Databricks reward.py) for Factual Accuracy

In [ ]:
!wget -q https://raw.githubusercontent.com/databricks/officeqa/main/reward.py -O reward.py
from reward import score_answer
print("score_answer imported.")

## 11. Full pipeline runner

One function, fed by `BASELINE` or `ENGINEERED`, produces every metric in the scorecard.

In [ ]:
from tqdm import tqdm

def run_pipeline(cfg: RunConfig, questions_df):
    print(f"\n=== Running {cfg.name.upper()} ===")

    chunker = Chunker(cfg)
    chunks = chunker.build(bulletin_files)
    print(f"{cfg.name}: {len(chunks)} chunks")

    index = VectorIndex(embedder).build(chunks)

    retrieval_metrics = evaluate_retrieval(index, cfg, questions_df)
    print(f"Hit Rate@{cfg.top_k}: {retrieval_metrics['hit_rate']:.2%} | MRR: {retrieval_metrics['mrr']:.4f}")

    gen_rows = []
    total_claims, supported_claims, answered = 0, 0, 0

    for _, row in tqdm(questions_df.iterrows(), total=len(questions_df)):
        year, month = (parse_query_time(row['question']) if cfg.use_metadata else (None, None))
        retrieved = index.search(row['question'], k=cfg.top_k, year_filter=year, month_filter=month)
        context = "\n\n".join(f"[Source: {r['meta']['source']}]\n{r['text']}" for r in retrieved)

        prompt = PROMPT_TEMPLATE.format(context=context[:8000], question=row['question'])
        raw, model_used = call_gemini(prompt)
        answer = extract_answer(raw)

        try:
            correct = score_answer(str(row['answer']), answer, tolerance=0.01)
        except Exception:
            correct = False

        g = check_groundedness(answer, context)
        if g is not None:
            answered += 1
            supported_claims += g[0]
            total_claims += g[1]

        gen_rows.append({
            "uid": row['uid'], "difficulty": row['difficulty'],
            "ground_truth": row['answer'], "prediction": answer,
            "correct": correct, "model": model_used,
        })
        time.sleep(1)

    gen_df = pd.DataFrame(gen_rows)
    factual_accuracy = gen_df['correct'].sum() / len(gen_df)
    groundedness = (supported_claims / total_claims) if total_claims else 0.0
    hallucination_rate = 1 - groundedness if total_claims else 0.0

    return {
        "config": cfg,
        "hit_rate": retrieval_metrics['hit_rate'],
        "mrr": retrieval_metrics['mrr'],
        "groundedness": groundedness,
        "factual_accuracy": factual_accuracy,
        "hallucination_rate": hallucination_rate,
        "answered_questions": answered,
        "predictions": gen_df,
    }


## 12. Run both configs

In [ ]:
baseline_results = run_pipeline(BASELINE, df_questions)
engineered_results = run_pipeline(ENGINEERED, df_questions)

baseline_results["predictions"].to_csv("./data/baseline_predictions.csv", index=False)
engineered_results["predictions"].to_csv("./data/engineered_predictions.csv", index=False)


## 13. Scorecard

In [ ]:
def pct(x): return f"{x:.2%}"

scorecard = pd.DataFrame({
    "Metric": ["Hit Rate@5", "MRR", "Groundedness", "Factual Accuracy", "Hallucination Rate"],
    "Baseline": [
        pct(baseline_results["hit_rate"]), f"{baseline_results['mrr']:.4f}",
        pct(baseline_results["groundedness"]), pct(baseline_results["factual_accuracy"]),
        pct(baseline_results["hallucination_rate"]),
    ],
    "Engineered": [
        pct(engineered_results["hit_rate"]), f"{engineered_results['mrr']:.4f}",
        pct(engineered_results["groundedness"]), pct(engineered_results["factual_accuracy"]),
        pct(engineered_results["hallucination_rate"]),
    ],
})
scorecard


## 14. Reflection

**The Bottleneck.** Looking at the Baseline scorecard, compare Hit Rate@5 against
Groundedness/Factual Accuracy. A low Hit Rate with tiny, non-overlapping 400-char
chunks points to a **retriever** problem: the "Librarian" isn't handing the "Student"
the right page, so no downstream prompt engineering can fix the answer.

**The Metadata Fix.** Adding `year`/`month`/`quarter` tags and pre-filtering the FAISS
search to the matching subset should move Hit Rate@5 and MRR the most — it removes
cross-year noise from the *search space itself*. Groundedness/Factual Accuracy improve
too, but only as a downstream consequence of the LLM finally seeing the right chunk;
they don't move on their own without a retrieval fix.

**Scaling Insight.** Going from an 8-year subset to the full 1939–2025 archive, the
first thing to break is the **flat FAISS index** (`IndexFlatIP`) — it's O(n) per
query with no compression, so both memory and query latency scale linearly with
83 years of bulletins. At that scale you'd want an IVF/HNSW index (or a managed
vector DB with sharding) plus coarser metadata partitioning (e.g., per-decade
collections) so a single query doesn't have to touch the entire corpus.
